# Milestone M6–M7: Eksperimen Utama CNN Text Classifier & Evaluasi Tahap 1
## Kelompok 4 — CNN for Text Classification (Indonesian Hate Speech Detection)

**Mata Kuliah:** Workshop Proyek Sistem Cerdas 2026  
**Arsitektur:** Yoon Kim (2014) Multi-kernel Conv1D (Filter sizes: [3, 4, 5])  

### Tujuan Notebook:
1. Memuat dataset split sequence atau tokenizer terkonfigurasi.
2. Menginisialisasi arsitektur **CNNTextClassifier** dengan multi-kernel Conv1D, GlobalMaxPooling, Dropout, dan Dense.
3. Menjalankan training dengan penanganan class imbalance (`ImbalanceHandler`) melalui `ModelTrainer`.
4. Melakukan evaluasi mendalam pada test set (Macro-F1, Precision, Recall, Confusion Matrix) menggunakan `MetricCalculator` & `ConfusionMatrixPlotter`.
5. Menjalankan modul **Error Analysis** (`ErrorAnalyzer`) untuk membedah kasus False Positive & False Negative.
6. Melakukan visualisasi atribusi kata dengan **Saliency Mapper** (`SaliencyMapper`).

In [ ]:
import sys
from pathlib import Path
import os

# Tambahkan root project ke sys.path
ROOT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.utils.config import Config
from src.utils.seed import set_seed
from src.models.cnn_model import CNNTextClassifier
from src.training.trainer import ModelTrainer
from src.training.imbalance import ImbalanceHandler
from src.preprocessing.tokenizer import TextTokenizer
from src.preprocessing.padder import SequencePadder
from src.evaluation.metrics import MetricCalculator
from src.evaluation.confusion import ConfusionMatrixPlotter
from src.evaluation.error_analysis import ErrorAnalyzer
from src.explainability.saliency import SaliencyMapper

set_seed(Config.SEED)
print(f"Project root: {ROOT_DIR}")
print(f"Arsitektur Filter Sizes: {Config.FILTER_SIZES}, Filters: {Config.NUM_FILTERS}")

### 1. Persiapan Data Train, Validation, dan Test

In [ ]:
train_path = ROOT_DIR / Config.TRAIN_CSV
val_path = ROOT_DIR / Config.VAL_CSV
test_path = ROOT_DIR / Config.TEST_CSV

if os.path.exists(train_path) and os.path.exists(val_path) and os.path.exists(test_path):
    train_df = pd.read_csv(train_path)
    val_df = pd.read_csv(val_path)
    test_df = pd.read_csv(test_path)
else:
    print("Menggunakan mock sequence data untuk verifikasi graf...")
    train_df = pd.DataFrame({
        "text_clean": ["dasar provokator busuk", "selamat pagi rekan semua", "hajar penjahat itu", "terima kasih infonya"] * 10,
        "label": [1, 0, 1, 0] * 10
    })
    val_df = pd.DataFrame({
        "text_clean": ["mampus lu penipu", "semoga sukses selalu"] * 5,
        "label": [1, 0] * 5
    })
    test_df = pd.DataFrame({
        "text_clean": ["dasar kadrun bangsat", "selamat menikmati liburan"] * 5,
        "label": [1, 0] * 5
    })

tok_path = ROOT_DIR / Config.TOKENIZER_OUTPUT
if os.path.exists(tok_path):
    tokenizer = TextTokenizer.load(str(tok_path))
else:
    tokenizer = TextTokenizer(vocab_size=Config.VOCAB_SIZE, oov_token=Config.OOV_TOKEN)
    tokenizer.fit(train_df["text_clean"].astype(str).tolist())

padder = SequencePadder(max_len=Config.MAX_LEN)

X_train = padder.pad(tokenizer.texts_to_sequences(train_df["text_clean"].astype(str).tolist()))
y_train = train_df["label"].values

X_val = padder.pad(tokenizer.texts_to_sequences(val_df["text_clean"].astype(str).tolist()))
y_val = val_df["label"].values

X_test = padder.pad(tokenizer.texts_to_sequences(test_df["text_clean"].astype(str).tolist()))
y_test = test_df["label"].values

print(f"X_train: {X_train.shape}, X_val: {X_val.shape}, X_test: {X_test.shape}")

### 2. Inisialisasi & Training CNN Multi-kernel

In [ ]:
# Inisialisasi model CNN
cnn_model = CNNTextClassifier(config=Config)
cnn_model.build_model()

# Inisialisasi handler imbalance & trainer
imbalance_handler = ImbalanceHandler(strategy=Config.IMBALANCE_STRATEGY)
trainer = ModelTrainer(model=cnn_model, imbalance_handler=imbalance_handler)

# Eksekusi training
history = trainer.fit(X_train=X_train, y_train=y_train, X_val=X_val, y_val=y_val)
print("Training completed.")

### 3. Evaluasi Performa pada Test Set

In [ ]:
y_pred = cnn_model.predict(X_test, threshold=0.5)
y_prob = cnn_model.predict_proba(X_test)

metric_calc = MetricCalculator()
metrics_cnn = metric_calc.compute_all(y_test, y_pred)

print("=== Hasil Evaluasi CNN Multi-kernel ===")
print(f"Macro-F1   : {metrics_cnn['macro_f1']:.4f}")
print(f"Macro-Prec : {metrics_cnn['precision_macro']:.4f}")
print(f"Macro-Rec  : {metrics_cnn['recall_macro']:.4f}")
print(f"Accuracy   : {metrics_cnn['accuracy']:.4f}")
print("Per Class  :", metrics_cnn["per_class"])

# Simpan metrik JSON
metrics_out_path = ROOT_DIR / "outputs/metrics/cnn_stage1_metrics.json"
metric_calc.save_metrics(metrics_cnn, str(metrics_out_path))
print(f"Metrik disimpan ke: {metrics_out_path}")

### 4. Visualisasi Confusion Matrix

In [ ]:
cm_plotter = ConfusionMatrixPlotter(class_names=["Non-toxic", "Toxic"])
cm_out_path = ROOT_DIR / "outputs/plots/confusion_matrix_cnn_stage1.png"
cm_plotter.plot_and_save(y_test, y_pred, output_path=str(cm_out_path), title="Confusion Matrix: CNN Text Classifier (Stage 1)")
print(f"Plot Confusion Matrix tersimpan di: {cm_out_path}")

### 5. Error Analysis (False Positives & False Negatives)

In [ ]:
test_df["pred_label"] = y_pred
test_df["pred_prob"] = y_prob[:, 0] if y_prob.ndim > 1 else y_prob

error_analyzer = ErrorAnalyzer()
analysis_result = error_analyzer.analyze(
    df=test_df,
    text_col="text_clean",
    true_label_col="label",
    pred_label_col="pred_label",
    prob_col="pred_prob",
    top_n=5
)

print("=== Ringkasan Error Analysis ===")
print(f"Total Sampel Uji     : {analysis_result['total_samples']}")
print(f"False Positives (FP) : {analysis_result['false_positives_count']}")
print(f"False Negatives (FN) : {analysis_result['false_negatives_count']}")

error_out_path = ROOT_DIR / "outputs/metrics/error_analysis_stage1.json"
error_analyzer.save_analysis(analysis_result, str(error_out_path))
print(f"Hasil analisis error disimpan ke: {error_out_path}")

### 6. Explainability via Saliency Mapper

In [ ]:
saliency_mapper = SaliencyMapper(model=cnn_model, tokenizer=tokenizer, padder=padder)

sample_text = "dasar provokator busuk perusak bangsa"
word_scores = saliency_mapper.compute_word_saliency(sample_text)

print(f"Teks Uji: '{sample_text}'")
print("Atribusi Bobot Kata (Saliency):")
for word, score in word_scores:
    bar = "█" * int(score * 20)
    print(f"{word:<15} | {score:.4f} | {bar}")

### 7. Penyimpanan Model Terlatih

In [ ]:
os.makedirs(ROOT_DIR / "outputs/models", exist_ok=True)
cnn_model.save(str(ROOT_DIR / Config.MODEL_OUTPUT))
print(f"Model artefak tersimpan di: {Config.MODEL_OUTPUT}")

### 8. Kesimpulan Milestone M6–M7
1. Model CNN Multi-kernel Yoon Kim berhasil diinisialisasi dan dilatih dengan penanganan class imbalance.
2. Evaluasi metrik dan confusion matrix tersimpan ke folder `outputs/`.
3. Saliency mapping dan error analysis siap dijadikan bahan demonstrasi UTS (M8) dan prototype Streamlit.